In [ ]:
!pip install openai gradio plotly -q
!apt-get install fonts-nanum -y -q
print("설치 완료 ✅")

Reading package lists...
Building dependency tree...
Reading state information...
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
설치 완료 ✅


In [ ]:
import pandas as pd
import sqlite3
import json
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'NanumGothic'

df = pd.read_csv('/content/lazada-products.csv')
print(df.shape)
df.head()

(1000, 29)


,url,title,rating,reviews,initial_price,final_price,currency,image,seller_name,breadcrumb,...,color,returns_and_warranty,is_super_seller,promotions,brand,product_variation,lazmall,domain,number_sold,gmv
0,https://www.lazada.co.id/products/dioda-damper...,DIODA DAMPER DMV 1500 TV POLYTRON NEW ORIGINAL...,4.6,27,0.0,10000.0,IDR,"[""https://img.lazcdn.com/g/ff/kf/Sc92f4b3c99c0...",YUDIANA YUDI SPERPAT ELEKTRONIK,"[""Televisi & Video"",""Televisi Digital""]",...,NaN,"[""Berubah Pikiran"",""7 Hari Gratis Pengembalian...",False,[],TR,[],False,https://www.lazada.co.id,111,1110000.0
1,https://www.lazada.co.id/products/victus-lapto...,Victus Laptop Gaming HP AMD Ryzen 5 NVIDIA GeF...,0.0,0,17555000.0,16499000.0,IDR,"[""https://img.lazcdn.com/g/p/3440632c069eddd82...",HP,"[""Komputer & Laptop"",""Laptop"",""Laptop Umum""]",...,NaN,"[""Produk Original"",""Berubah Pikiran"",""30 Hari ...",True,[],HP,"[{""name"":""Varian"",""value"":""15-fb2666AX""}]",True,https://www.lazada.co.id,0,0.0
2,https://www.lazada.co.id/products/laptop-hp-15...,Laptop HP 15s-fq5148TU Core i3 UHD 4GB & 8GB R...,5.0,47,6499000.0,6099000.0,IDR,"[""https://img.lazcdn.com/g/p/23782d5f1bd37a89c...",HP,"[""Komputer & Laptop"",""Laptop"",""Laptop Umum""]",...,14s-dq5115TU,"[""Produk Original"",""Berubah Pikiran"",""30 Hari ...",True,[],HP,"[{""name"":""Warna"",""value"":""14s-dq5115TU""}]",True,https://www.lazada.co.id,91,555009000.0
3,https://www.lazada.co.id/products/printer-hp-d...,Printer HP DeskJet 2336 All in One ( Print Sca...,5.0,177,990000.0,880000.0,IDR,"[""https://img.lazcdn.com/g/p/bf68d11f46c85761a...",HP,"[""Pencetak & Monitor"",""Printer"",""Printer Ink J...",...,2336 Putih,"[""Produk Original"",""Berubah Pikiran"",""30 Hari ...",True,[],HP,"[{""name"":""Warna"",""value"":""2336 Putih""}]",True,https://www.lazada.co.id,409,359920000.0
4,https://www.lazada.co.id/products/roda-koper-r...,"RODA KOPER, RODA PENGGANTI, RODA DOUBLE WHEEL,...",5.0,2,0.0,65000.0,IDR,"[""https://img.lazcdn.com/g/p/abc4f89073ba5380c...",dewi05588,"[""Tas & Travel"",""Tas Anak"",""Koper""]",...,NaN,"[""Berubah Pikiran"",""7 Hari Gratis Pengembalian...",False,[],Tidak Ada Merk,"[{""name"":""RODA"",""value"":""AUDI""}]",False,https://www.lazada.co.id,2,130000.0


In [ ]:
print("=== 정제 전 데이터 현황 ===")
print(f"총 행수: {len(df)}")
print(f"총 컬럼수: {len(df.columns)}")
print()

print("【자료형】")
print(df.dtypes)
print()

print("【결측치】")
print(df.isnull().sum())
print()

print("【가격 이상치】")
print(df['final_price'].describe())
print(f"0원 이하 상품 수: {(df['final_price'] <= 0).sum()}")
print()

print(f"【평점 0인 상품 수】: {(df['rating'] == 0).sum()}")
print(f"【고유 셀러 수】: {df['seller_name'].nunique()}")
print(f"【고유 브랜드 수】: {df['brand'].nunique()}")
print()

print("【breadcrumb 샘플】")
print(df['breadcrumb'].head(3).values)

=== 정제 전 데이터 현황 ===
총 행수: 1000
총 컬럼수: 29

【자료형】
url                        object
title                      object
rating                    float64
reviews                     int64
initial_price             float64
final_price               float64
currency                   object
image                      object
seller_name                object
breadcrumb                 object
product_specifications     object
product_description        object
seller_ratings            float64
seller_ship_on_time        object
seller_chat_response       object
sku                        object
mpn                         int64
colors                     object
variations                 object
color                      object
returns_and_warranty       object
is_super_seller              bool
promotions                 object
brand                      object
product_variation          object
lazmall                      bool
domain                     object
number_sold                 int64


In [ ]:
print(f"정제 전: {len(df)}건")

# 1. 평점 0 → NULL
df['rating'] = df['rating'].replace(0, None)

# 2. 결측치 처리 (텍스트 컬럼)
text_cols = ['colors', 'color', 'seller_ratings',
             'seller_ship_on_time', 'seller_chat_response']
df[text_cols] = df[text_cols].fillna('없음')

# 3. breadcrumb 파싱 → level1 / level2 / level3
import json

def parse_breadcrumb(bc_str):
    try:
        arr = json.loads(bc_str)
        return {
            'level1': arr[0] if len(arr) > 0 else None,
            'level2': arr[1] if len(arr) > 1 else None,
            'level3': arr[2] if len(arr) > 2 else None,
        }
    except:
        return {'level1': None, 'level2': None, 'level3': None}

cat_df = df['breadcrumb'].apply(parse_breadcrumb).apply(pd.Series)
df = pd.concat([df, cat_df], axis=1)

print(f"정제 후: {len(df)}건")
print(f"평점 NULL 처리 확인: {df['rating'].isnull().sum()}개")
print(f"level1 샘플: {df['level1'].value_counts().head(5)}")

정제 전: 1000건
정제 후: 1000건
평점 NULL 처리 확인: 296개
level1 샘플: level1
Mobiles & Tablets          197
Aksesoris Elektronik       109
Electronics Accessories    105
Beauty                      98
Televisions & Videos        95
Name: count, dtype: int64


In [ ]:
import os
import sqlite3

# 기존 DB 삭제 후 새로 생성
if os.path.exists('lazada.db'):
    os.remove('lazada.db')

conn = sqlite3.connect('lazada.db')
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
CREATE TABLE IF NOT EXISTS sellers (
    id                   INTEGER PRIMARY KEY AUTOINCREMENT,
    seller_name          TEXT    NOT NULL UNIQUE,
    seller_ratings       TEXT,
    seller_ship_on_time  TEXT,
    seller_chat_response TEXT,
    is_super_seller      INTEGER,
    is_lazmall           INTEGER
);

CREATE TABLE IF NOT EXISTS brands (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    brand_name TEXT    NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS categories (
    id     INTEGER PRIMARY KEY AUTOINCREMENT,
    level1 TEXT,
    level2 TEXT,
    level3 TEXT
);

CREATE TABLE IF NOT EXISTS products (
    id                   INTEGER PRIMARY KEY AUTOINCREMENT,
    seller_id            INTEGER NOT NULL REFERENCES sellers(id),
    brand_id             INTEGER NOT NULL REFERENCES brands(id),
    category_id          INTEGER NOT NULL REFERENCES categories(id),
    sku                  TEXT    UNIQUE,
    title                TEXT,
    rating               REAL    CHECK (rating BETWEEN 0 AND 5),
    reviews              INTEGER,
    initial_price        REAL    CHECK (initial_price >= 0),
    final_price          REAL    CHECK (final_price >= 0),
    currency             TEXT,
    number_sold          INTEGER,
    gmv                  REAL,
    product_description  TEXT,
    returns_and_warranty TEXT,
    promotions           TEXT,
    url                  TEXT
);
""")

conn.commit()
print("DB 생성 완료 ✅")

DB 생성 완료 ✅


In [ ]:
# 1. sellers 적재
sellers_df = df[['seller_name', 'seller_ratings', 'seller_ship_on_time',
                  'seller_chat_response', 'is_super_seller', 'lazmall']].drop_duplicates(subset='seller_name')
sellers_df = sellers_df.rename(columns={'lazmall': 'is_lazmall'})
sellers_df.to_sql('sellers', conn, if_exists='append', index=False)
print(f"sellers 적재: {len(sellers_df)}건")

# 2. brands 적재
brands_df = df[['brand']].drop_duplicates().rename(columns={'brand': 'brand_name'})
brands_df = brands_df[brands_df['brand_name'].notna()]
brands_df.to_sql('brands', conn, if_exists='append', index=False)
print(f"brands 적재: {len(brands_df)}건")

# 3. categories 적재
categories_df = df[['level1', 'level2', 'level3']].drop_duplicates()
categories_df.to_sql('categories', conn, if_exists='append', index=False)
print(f"categories 적재: {len(categories_df)}건")

sellers 적재: 197건
brands 적재: 78건
categories 적재: 66건


In [ ]:
# FK 매핑 (이름 → ID 변환)
seller_map = pd.read_sql("SELECT id AS seller_id, seller_name FROM sellers", conn)
brand_map = pd.read_sql("SELECT id AS brand_id, brand_name FROM brands", conn)
category_map = pd.read_sql("SELECT id AS category_id, level1, level2, level3 FROM categories", conn)

df_p = df.merge(seller_map, on='seller_name', how='left')
df_p = df_p.merge(brand_map, left_on='brand', right_on='brand_name', how='left')
df_p = df_p.merge(category_map, on=['level1', 'level2', 'level3'], how='left')

# products 적재
products_df = df_p[[
    'seller_id', 'brand_id', 'category_id',
    'sku', 'title', 'rating', 'reviews',
    'initial_price', 'final_price', 'currency',
    'number_sold', 'gmv', 'product_description',
    'returns_and_warranty', 'promotions', 'url'
]]

products_df.to_sql('products', conn, if_exists='append', index=False)
print(f"products 적재: {len(products_df)}건")

# 검증
for table in ['sellers', 'brands', 'categories', 'products']:
    count = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table}", conn)['cnt'][0]
    print(f"{table}: {count}건")

products 적재: 1000건
sellers: 197건
brands: 78건
categories: 66건
products: 1000건


In [ ]:
import gradio as gr
import plotly.express as px

# ── SQL 데이터 추출 ──────────────────────

# KPI
total_products = pd.read_sql("SELECT COUNT(*) AS cnt FROM products", conn)['cnt'][0]
total_sellers  = pd.read_sql("SELECT COUNT(*) AS cnt FROM sellers", conn)['cnt'][0]
total_gmv      = pd.read_sql("SELECT SUM(gmv) AS s FROM products", conn)['s'][0]
avg_rating     = pd.read_sql("SELECT ROUND(AVG(rating),2) AS r FROM products WHERE rating IS NOT NULL", conn)['r'][0]

# 차트1: 카테고리별 매출 TOP 10
cat_gmv = pd.read_sql("""
    SELECT c.level1 AS 카테고리, SUM(p.gmv) AS 총매출
    FROM products p JOIN categories c ON p.category_id = c.id
    GROUP BY c.level1 ORDER BY 총매출 DESC LIMIT 10
""", conn)

# 차트2: 슈퍼셀러 vs 일반셀러
super_compare = pd.read_sql("""
    SELECT
        CASE WHEN s.is_super_seller = 1 THEN '슈퍼셀러' ELSE '일반셀러' END AS 셀러유형,
        COUNT(p.id) AS 상품수,
        SUM(p.number_sold) AS 총판매수량,
        ROUND(AVG(p.rating), 2) AS 평균평점
    FROM products p JOIN sellers s ON p.seller_id = s.id
    GROUP BY s.is_super_seller
""", conn)

# 차트3: 평점 분포
rating_dist = pd.read_sql("""
    SELECT ROUND(rating, 1) AS 평점, COUNT(*) AS 상품수
    FROM products WHERE rating IS NOT NULL
    GROUP BY ROUND(rating, 1) ORDER BY 평점
""", conn)

# 차트4: 베스트셀러 TOP 10
bestseller = pd.read_sql("""
    SELECT p.title AS 상품명, s.seller_name AS 셀러,
           p.final_price AS 가격, p.number_sold AS 판매량, p.rating AS 평점
    FROM products p JOIN sellers s ON p.seller_id = s.id
    ORDER BY p.number_sold DESC LIMIT 10
""", conn)

# 차트5: 할인율 TOP 10
discount = pd.read_sql("""
    SELECT title AS 상품명,
           ROUND((initial_price - final_price) * 100.0 / initial_price, 1) AS 할인율
    FROM products
    WHERE initial_price > 0 AND initial_price > final_price
    ORDER BY 할인율 DESC LIMIT 10
""", conn)

print("데이터 추출 완료 ✅")

데이터 추출 완료 ✅


In [ ]:
import requests

# 실시간 환율 조회 (각 통화 → KRW)
response = requests.get("https://open.er-api.com/v6/latest/KRW")
data = response.json()

# KRW 기준 역산
rates_to_krw = {
    'IDR': data['rates']['IDR'],
    'MYR': data['rates']['MYR'],
    'PHP': data['rates']['PHP'],
    'SGD': data['rates']['SGD'],
    'THB': data['rates']['THB'],
}

print("현재 환율 (1 KRW 기준):")
for cur, rate in rates_to_krw.items():
    print(f"  1 KRW = {rate:.4f} {cur}  →  1 {cur} = {1/rate:,.2f} KRW")

# GMV를 통화별로 KRW 변환
thread_conn = sqlite3.connect('lazada.db')
gmv_by_currency = pd.read_sql("""
    SELECT currency, SUM(gmv) AS total_gmv
    FROM products
    GROUP BY currency
""", thread_conn)
thread_conn.close()

total_gmv_krw = 0
for _, row in gmv_by_currency.iterrows():
    cur = row['currency']
    if cur in rates_to_krw:
        total_gmv_krw += row['total_gmv'] / rates_to_krw[cur]

print(f"\n총 GMV (KRW): ₩ {total_gmv_krw:,.0f}")

현재 환율 (1 KRW 기준):
  1 KRW = 11.6558 IDR  →  1 IDR = 0.09 KRW
  1 KRW = 0.0027 MYR  →  1 MYR = 374.67 KRW
  1 KRW = 0.0397 PHP  →  1 PHP = 25.19 KRW
  1 KRW = 0.0008 SGD  →  1 SGD = 1,190.48 KRW
  1 KRW = 0.0216 THB  →  1 THB = 46.21 KRW

총 GMV (KRW): ₩ 30,706,712,603


In [ ]:
import os
from openai import OpenAI
import re

os.environ['OPENAI_API_KEY'] = "여기에_API_키_입력"  # Colab 비밀키 사용 권장: userdata.get("OPENAI_API_KEY")
client = OpenAI()

# 스키마 정보
schema_info = """
[테이블 스키마]
products (id, sku, title, initial_price, final_price, rating, reviews,
          number_sold, gmv, currency, product_description, returns_and_warranty,
          promotions, url, seller_id FK→sellers, brand_id FK→brands, category_id FK→categories)
sellers (id, seller_name, seller_ratings, seller_ship_on_time, seller_chat_response, is_super_seller, is_lazmall)
brands (id, brand_name)
categories (id, level1, level2, level3)
"""

system_prompt = f"""
당신은 SQLite SQL 전문가입니다.
{schema_info}
[규칙]
1. SQLite 문법만 사용
2. SELECT 쿼리만 작성
3. 만약 사용자가 삭제, 수정, 드롭, 초기화 등 데이터를 변경하는 요청을 하면 반드시 "BLOCKED" 한 단어만 출력
4. JOIN이 필요하면 INNER JOIN 사용
5. 결과 컬럼에 한국어 alias 사용 (예: SUM(gmv) AS 총매출)
6. SQL 코드만 출력하고 설명이나 주석 추가 금지
7. 반드시 LIMIT 20 이하로 제한
"""

def chat(message, history):
    try:
        sql = generate_sql(message)

        # BLOCKED 응답 처리
        if sql.strip().upper() == "BLOCKED":
            return history + [[message, "⚠️ 차단된 요청: 데이터 변경은 허용되지 않습니다"]]

        safe, reason = is_safe_sql(sql)
        if not safe:
            return history + [[message, f"⚠️ 차단된 요청: {reason}"]]

        thread_conn = sqlite3.connect('lazada.db')
        df_result = pd.read_sql(sql, thread_conn)
        thread_conn.close()

        summary = summarize_result(message, sql, df_result)
        answer = f"{summary}\n\n**생성된 SQL:**\n```sql\n{sql}\n```\n\n**결과 ({len(df_result)}건):**\n{df_result.to_markdown(index=False)}"
    except Exception as e:
        answer = f"❌ 오류 발생: {str(e)}"

    return history + [[message, answer]]

# 가드레일
DANGEROUS = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'CREATE', 'GRANT']

def is_safe_sql(sql):
    upper = sql.upper()
    if not upper.strip().startswith('SELECT'):
        return False, "SELECT 쿼리만 허용됩니다"
    for kw in DANGEROUS:
        if re.search(rf'\b{kw}\b', upper):
            return False, f"금지된 명령어 포함: {kw}"
    return True, "OK"

# SQL 생성
def generate_sql(question):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0,
        max_tokens=500
    )
    sql = response.choices[0].message.content.strip()
    # 코드블록 제거
    sql = re.sub(r'```sql|```', '', sql).strip()
    return sql

# 결과 요약
def summarize_result(question, sql, df_result):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 데이터 분석 결과를 친절하게 요약해주는 assistant입니다. 한국어로 2~3문장으로 요약하세요."},
            {"role": "user", "content": f"질문: {question}\nSQL: {sql}\n결과: {df_result.to_string()}"}
        ],
        temperature=0.3,
        max_tokens=300
    )
    return response.choices[0].message.content.strip()

# 챗봇 함수
def chat(message, history):
    try:
        sql = generate_sql(message)

        if sql.strip().upper() == "BLOCKED":
            return history + [[message, "⚠️ 차단된 요청: 데이터 변경은 허용되지 않습니다"]]

        safe, reason = is_safe_sql(sql)
        if not safe:
            return history + [[message, f"⚠️ 차단된 요청: {reason}"]]

        # 스레드 안에서 새 연결 생성
        thread_conn = sqlite3.connect('lazada.db')
        df_result = pd.read_sql(sql, thread_conn)
        thread_conn.close()

        summary = summarize_result(message, sql, df_result)
        answer = f"{summary}\n\n**생성된 SQL:**\n```sql\n{sql}\n```\n\n**결과 ({len(df_result)}건):**\n{df_result.to_markdown(index=False)}"
    except Exception as e:
        answer = f"❌ 오류 발생: {str(e)}"

    return history + [[message, answer]]

# 대시보드 + 챗봇 통합
def build_dashboard():
    fig1 = px.bar(cat_gmv, x='총매출', y='카테고리', orientation='h',
                  title='카테고리별 매출 TOP 10')
    fig1.update_traces(marker_color='#4f8ef7')
    fig1.update_layout(yaxis={'categoryorder': 'total ascending'})

    fig2 = px.bar(super_compare.melt(id_vars='셀러유형', value_vars=['상품수', '총판매수량', '평균평점']),
                  x='셀러유형', y='value', color='variable', barmode='group',
                  title='슈퍼셀러 vs 일반셀러 비교',
                  facet_col='variable',
                  color_discrete_map={'상품수': '#4f8ef7', '총판매수량': '#f59e0b', '평균평점': '#22c55e'})
    fig2.update_yaxes(matches=None)

    fig3 = px.bar(rating_dist, x='평점', y='상품수', title='평점 분포')
    fig3.update_traces(marker_color='#22c55e')

    discount['상품명_short'] = discount['상품명'].str[:30] + '...'
    fig4 = px.bar(discount, x='할인율', y='상품명_short', orientation='h',
                  title='할인율 TOP 10')
    fig4.update_traces(marker_color='#f87171')
    fig4.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)

    return fig1, fig2, fig3, fig4

with gr.Blocks(title="Lazada 관리자 대시보드") as demo:
    gr.Markdown("# 🛒 Lazada 관리자 대시보드")

    with gr.Tabs():
        with gr.Tab("📊 대시보드"):
            with gr.Row():
                gr.Number(value=total_products, label="총 상품 수")
                gr.Number(value=total_sellers,  label="총 셀러 수")
                gr.Textbox(value=f"₩ {total_gmv_krw:,.0f}", label="총 GMV (KRW)", interactive=False)
                gr.Number(value=avg_rating,     label="평균 평점")

            with gr.Row():
                gr.Plot(value=build_dashboard()[0])
                gr.Plot(value=build_dashboard()[1])

            with gr.Row():
                gr.Plot(value=build_dashboard()[2])
                gr.Plot(value=build_dashboard()[3])

            gr.Markdown("### 🏆 베스트셀러 TOP 10")
            gr.Dataframe(value=bestseller, wrap=False)

        with gr.Tab("💬 AI 데이터 챗봇"):
            gr.Markdown("### 🤖 자연어로 데이터를 조회하세요")
            chatbot = gr.Chatbot(height=400)
            msg = gr.Textbox(label="질문 입력", placeholder="예: 가장 비싼 상품 5개 보여줘")
            with gr.Row():
                send_btn = gr.Button("전송", variant="primary")
                clear_btn = gr.Button("초기화")

            gr.Markdown("**예시 질문:**")
            with gr.Row():
                gr.Button("가장 비싼 상품 5개").click(lambda: "가장 비싼 상품 5개 보여줘", outputs=msg)
                gr.Button("슈퍼셀러 매출 TOP5").click(lambda: "슈퍼셀러 중 매출이 가장 높은 셀러 5명은?", outputs=msg)
                gr.Button("Xiaomi 브랜드 통계").click(lambda: "Xiaomi 브랜드 상품의 평균 가격과 총 판매수량", outputs=msg)

            send_btn.click(chat, inputs=[msg, chatbot], outputs=chatbot)
            clear_btn.click(lambda: [], outputs=chatbot)

demo.launch(share=True)